In [17]:
!pip3 install gtfs-realtime-bindings
!pip3 install python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [35]:
from google.transit import gtfs_realtime_pb2
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

In [11]:
load_dotenv()

API_KEY = os.getenv("MTA_API_KEY")

The General Transit Feed Specification is an open standard used to distribute relevant information about transit systems.

It contains two parts, the GTFS schedule and the GTFS realtime.

Given the steady stream of L trains coming in the and delays that often plague the MTA, I will focus on real time.

In [14]:
# https://api.mta.info/#/subwayRealTimeFeeds
L_TRAIN_FEED_URL = f"https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-l"

# Create GTFS Realtime object
feed = gtfs_realtime_pb2.FeedMessage()

# Fetch data
response = requests.get(L_TRAIN_FEED_URL)
feed.ParseFromString(response.content)

31373

Vehicle position is used to provide automatically generated information on the location of a vehicle, such as from a GPS device on board. A single vehicle position should be provided for every vehicle that is capable of providing it.

The trip that the vehicle is currently serving should be given through a TripDescriptor. You can also provide a VehicleDescriptor, which specifies a precise physical vehicle that you are providing updates about. Documentation is provided below.

A timestamp denoting the time when the position reading was taken can be provided. Note that this is different from the timestamp in the feed header, which is the time that this message was generated by the server.

Current passage can also be provided (either as a stop_sequence or stop_id). This is a reference to the stop that the vehicle is either on its way to, or already stopped at.

We know from the official MTA GTFS static feed that is available zipped in the documentation that the `stop_id` we are most interested in is L14.

For directional reasons, if we are going into Manhattan, we want L14N (I believe westbound is more accurate but I digress). Vice versa, we want L14S.

In [37]:
morgan_stops = ['L14', 'L14N', 'L14S']

manhattan_bound = []
rockaway_bound = []

for entity in feed.entity:
	if entity.HasField('trip_update'):
		trip = entity.trip_update
		if trip.trip.route_id == 'L':
			for stop in trip.stop_time_update:
				if stop.stop_id in morgan_stops:
					arrival_ts = stop.arrival.time if stop.HasField('arrival') else None
					departure_ts = stop.departure.time if stop.HasField('departure') else None

					# Yes, I am a military time believer.
					arrival_time = datetime.fromtimestamp(arrival_ts).strftime("%H:%M:%S") if arrival_ts else "N/A"
					departure_time = datetime.fromtimestamp(departure_ts).strftime("%H:%M:%S") if departure_ts else "N/A"

					print(f"Morgan Av stop_id {stop.stop_id}: Arrival {arrival_time}, Departure {departure_time}")

					if stop.stop_id == 'L14N':
						manhattan_bound.append(departure_time)
					elif stop.stop_id == 'L14S':
						rockaway_bound.append(departure_time)

Morgan Av stop_id L14S: Arrival 20:56:23, Departure 20:56:38
Morgan Av stop_id L14N: Arrival 21:00:45, Departure 21:01:15
Morgan Av stop_id L14S: Arrival 20:58:59, Departure 20:59:14
Morgan Av stop_id L14S: Arrival 21:00:40, Departure 21:01:10
Morgan Av stop_id L14N: Arrival 21:05:30, Departure 21:06:00
Morgan Av stop_id L14S: Arrival 21:02:45, Departure 21:03:15
Morgan Av stop_id L14N: Arrival 21:09:38, Departure 21:10:08
Morgan Av stop_id L14S: Arrival 21:05:22, Departure 21:05:52
Morgan Av stop_id L14N: Arrival 21:12:46, Departure 21:13:16
Morgan Av stop_id L14S: Arrival 21:09:00, Departure 21:09:30
Morgan Av stop_id L14N: Arrival 21:17:30, Departure 21:18:00
Morgan Av stop_id L14S: Arrival 21:13:00, Departure 21:13:30
Morgan Av stop_id L14N: Arrival 21:21:30, Departure 21:22:00
Morgan Av stop_id L14S: Arrival 21:17:00, Departure 21:17:30
Morgan Av stop_id L14N: Arrival 21:25:30, Departure 21:26:00
Morgan Av stop_id L14S: Arrival 21:21:00, Departure 21:21:30
Morgan Av stop_id L14N: 

In [40]:
# Next three trains headed to Manhattan (at the time of running the fetch data cell)
print(manhattan_bound[:3])

# Next three trains headed to Brooklyn
print(rockaway_bound[:3])

['21:01:15', '21:06:00', '21:10:08']
['20:56:38', '20:59:14', '21:01:10']
